# Code Explainator!

In [ ]:
# imports

import os
import requests
import re
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display
from pathlib import Path
from typing import List


In [ ]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')
grok_api_key = os.getenv('GROK_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if deepseek_api_key:
    print(f"DeepSeek API Key exists and begins {deepseek_api_key[:3]}")
else:
    print("DeepSeek API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

if grok_api_key:
    print(f"Grok API Key exists and begins {grok_api_key[:4]}")
else:
    print("Grok API Key not set (and this is optional)")

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:3]}")
else:
    print("OpenRouter API Key not set (and this is optional)")


In [ ]:
# Connect to OpenAI client library
# A thin wrapper around calls to HTTP endpoints

openai = OpenAI()

# For Gemini, DeepSeek and Groq, we can use the OpenAI python client
# Because Google and DeepSeek have endpoints compatible with OpenAI
# And OpenAI allows you to change the base_url

anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
deepseek_url = "https://api.deepseek.com"
groq_url = "https://api.groq.com/openai/v1"
grok_url = "https://api.x.ai/v1"
openrouter_url = "https://openrouter.ai/api/v1"
ollama_url = "http://localhost:11434/v1"

anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
deepseek = OpenAI(api_key=deepseek_api_key, base_url=deepseek_url)
groq = OpenAI(api_key=groq_api_key, base_url=groq_url)
grok = OpenAI(api_key=grok_api_key, base_url=grok_url)
openrouter = OpenAI(base_url=openrouter_url, api_key=openrouter_api_key)
ollama = OpenAI(api_key="ollama", base_url=ollama_url)

In [ ]:
def read_directory_contents(base_paths: List[str]) -> str:
    """
    Recursively reads all files under each path in base_paths and returns
    a single string containing relative file paths followed by their contents.

    :param base_paths: List of root directories to scan
    :return: Aggregated string of file paths and contents
    """
    result_parts = []

    for base_path in base_paths:
        for root, _, files in os.walk(base_path):
            for file_name in files:
                full_path = os.path.join(root, file_name)
                rel_path = os.path.relpath(full_path, base_path)

                try:
                    with open(full_path, 'r', encoding='utf-8', errors='ignore') as f:
                        content = f.read()
                except Exception as e:
                    # Skip unreadable files but record the issue
                    content = f"[Error reading file: {e}]"

                result_parts.append(
                    f"File directory: {rel_path}\n\n"
                    f"{content}\n"
                )

    return "\n".join(result_parts)

In [ ]:
allCode = read_directory_contents(["C:/Users/ME36352/MyFolder/REPOS/AI/claude-agents"])
print(allCode)

In [ ]:
claude_model = "claude-opus-4-8"
#gpt_model = "gpt-5.3-codex"
gpt_model = "gpt-5.5"
with open("taskDescription.txt", "r", encoding="utf-8") as f:
    task_description = f.read()
request = task_description
system_prompt = """

Analyze this workspace and create a detailed report of how it works. 
I want the relation of all the files and how they work. 
I want to know the logic of all files and how they are bing executed? Is this a python workspaces? 
How this workspace works? What is the purpose of it? Explain the whole workspace with full details. 

"""
#Reply in plain text. Don't use any markdown or other formatting. You can format it to look better, but don't use markdown.
#Just make it look good for readability.
system_prompt = system_prompt + allCode

In [ ]:
print(system_prompt)

In [ ]:
def call_claude(chat):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": chat}
    ]

    response = anthropic.chat.completions.create(
        model=claude_model,
        messages=messages,
        max_tokens=128000,                # REQUIRED for Anthropic
        extra_body={
            "thinking": {
                "type": "enabled",
                "budget_tokens": 126000   # must be < max_tokens
            }
        }
    )
    return response.choices[0].message.content

In [ ]:
def call_gemini(chat):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": chat}
    ]

    response = gemini.chat.completions.create(
        model="gemini-3.5-flash",
        messages=messages,
    )
    return response.choices[0].message.content

In [ ]:
llmResponse = call_claude(request)
print(llmResponse)

In [ ]:
llmResponse = call_gemini(request)
print(llmResponse)

In [ ]:
def call_llm(chat):
    messages = [
       {"role": "system", "content": system_prompt},
        {"role": "user", "content": chat}
    ]
    
    response = openai.chat.completions.create(model=gpt_model, messages=messages, reasoning_effort="high")
    return response.choices[0].message.content

In [ ]:
llmResponse = call_llm(request)
print(llmResponse)